In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from keras.models import Model
from keras.layers import Input, Dense
from lifelines.utils import concordance_index
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd /content/drive/MyDrive/BIPS

In [ ]:
def simulate_data(N, seed=None):
    np.random.seed(seed)
    x1 = np.random.normal(0, 1, N)
    x2 = np.random.normal(0, 1, N)
    x3 = np.random.normal(0, 1, N)
    x4 = np.random.normal(0, 1, N)
    x5 = np.random.normal(0, 1, N)
    x6 = np.random.normal(0, 1, N)
    x7 = np.random.binomial(1, 0.5, N)
    x8 = np.random.binomial(1, 0.5, N)
    x9 = np.random.binomial(1, 0.5, N)
    x10 = np.random.binomial(1, 0.5, N)
    x11 = np.random.binomial(1, 0.5, N)
    x12 = np.random.binomial(1, 0.5, N)
    rate1 = 0.01
    rate2 = 0.05
    b1 = np.log(2)
    b2 = 1.5
    betas1 = [b1, -b1, 0, 0, b1, -b1, b2, -b2, 0, 0, b2, -b2]
    betas2 = [0, 0, b1, -b1, b1, -b1, 0, 0, b2, -b2, b2, -b2]
    cens = np.random.exponential(1 / 0.01, N)
    lambda1 = rate1 * np.exp(betas1[0]*x1 + betas1[1]*x2 + betas1[2]*x3 + betas1[3]*x4 + betas1[4]*x5 + betas1[5]*x6 +
                            betas1[6]*x7 + betas1[7]*x8 + betas1[8]*x9 +
                            betas1[9]*x10 + betas1[10]*x11 + betas1[11]*x12)

    lambda2 = rate2 * np.exp(betas2[0]*x1 + betas2[1]*x2 + betas2[2]*x3 + betas2[3]*x4 + betas2[4]*x5 + betas2[5]*x6 +
                            betas2[6]*x7 + betas2[7]*x8 + betas2[8]*x9 +
                            betas2[9]*x10 + betas2[10]*x11 + betas2[11]*x12)

    h = lambda1 + lambda2
    dt = -np.log(np.random.uniform(size=N)) / h
    fcause = np.random.binomial(1, lambda1 / h)
    cause = np.where(fcause == 0, 2, 1)
    e = (dt <= cens) * cause
    dt = np.minimum(dt, cens)
    dat1 = pd.DataFrame({'dt': dt, 'e': e, 'x1': x1, 'x2': x2, 'x3': x3, 'x4': x4, 'x5': x5, 'x6': x6,
                         'x7': x7, 'x8': x8, 'x9': x9, 'x10': x10, 'x11': x11, 'x12': x12})

    return dat1

N = len(dat1)
index_train, index_test = train_test_split(range(N), test_size=0.2)

# Use iloc to select the rows by their positional index
train_D = dat1.iloc[index_train].reset_index(drop=True)
test_D = dat1.iloc[index_test].reset_index(drop=True)

X_train = train_D.drop(columns=['dt', 'e']).values
X_test = test_D.drop(columns=['dt', 'e']).values
t_train, t_test = train_D['dt'].values, test_D['dt'].values
e_train, e_test = train_D['e'].values, test_D['e'].values
y_true = np.column_stack((t_train, e_train))
def build_model():
    model = tf.keras.models.Sequential([
        tf.keras.layers.Flatten(input_shape=(12, )),
        tf.keras.layers.Dense(128, activation='selu'),
        tf.keras.layers.BatchNormalization(
            axis=-1, momentum=0.99, epsilon=0.001, center=True, scale=True,
            beta_initializer='zeros', gamma_initializer='ones',
            moving_mean_initializer='zeros',
            moving_variance_initializer='ones', beta_regularizer=None,
            gamma_regularizer=None, beta_constraint=None, gamma_constraint=None),
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(2)  # Two neurons for predicting hazards for event 1 and event 2
    ])
    return model

model = build_model()
def create_risk_sets(train_D, t_train, e_train, event_type):
    event_times = np.sort(t_train[e_train == event_type])
    rs = [np.where(t_train >= t)[0] for t in event_times]
    Risksets = [train_D.loc[x] for x in rs]
    Controls = [Risksets[z]['e'] == 0 for z in range(len(Risksets))]
    Ind = [control.index.values for control in Controls]
    event_time = np.where(e_train == event_type)[0]
    samples = [np.random.choice(control.index, size=8, replace=True) for control in Controls]
    rs = [np.append(sample, event_time[j]) for j, sample in enumerate(samples)]
    return rs, Risksets

def surv_likelihood(x_train, t_train, e_train, model, event_type):
    rs, Risksets = create_risk_sets(train_D, t_train, e_train, event_type)
    sort_idx = np.argsort(t_train)[::-1]
    exp_Xbeta = model(x_train[sort_idx, :])
    loss_values = []
    for i, riskset in enumerate(Risksets):
        temp = []
        for j in rs[i]:
            temp.append(tf.math.log(1 + tf.math.exp(-exp_Xbeta[i] + exp_Xbeta[j])))
        loss_values.append(tf.reduce_sum(temp))
    return tf.reduce_sum(loss_values)
def cox_loss(y_true, y_pred):
    loss_event1 = surv_likelihood(X_train, t_train, e_train, model, 1)
    loss_event2 = surv_likelihood(X_train, t_train, e_train, model, 2)
    return loss_event1 + loss_event2
#
model.compile(loss=cox_loss, optimizer='adam')
history = model.fit(X_train, y_true, epochs=100, batch_size=32, validation_split=0.2)
train_loss = history.history['loss']
val_loss = history.history['val_loss']
epochs = range(1, len(train_loss) + 1)
plt.plot(epochs, train_loss, 'b', label='Training Loss')
plt.plot(epochs, val_loss, 'r', label='Validation Loss')
plt.title('Training and Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()
def predict_cumulative_incidence_function(model, x_test, t_test, e_test, time_points):
    predictions = model.predict(x_test)
    cumulative_incidence_probs_event1 = []
    cumulative_incidence_probs_event2 = []

    for i in range(len(x_test)):
        individual_hazard1 = np.exp(predictions[i, 0])
        individual_hazard2 = np.exp(predictions[i, 1])
        cumulative_hazard1 = 0
        cumulative_hazard2 = 0
        cif_event1 = []
        cif_event2 = []
        survival_probability = 1

        for j, t in enumerate(time_points):
            if t <= t_test[i]:
                delta_t = t if j == 0 else (time_points[j] - time_points[j-1])

                cumulative_hazard1 += individual_hazard1 * delta_t
                cumulative_hazard2 += individual_hazard2 * delta_t

                cif_event1_value = survival_probability * (1 - np.exp(-individual_hazard1 * delta_t))
                cif_event2_value = survival_probability * (1 - np.exp(-individual_hazard2 * delta_t))

                survival_probability *= np.exp(-(individual_hazard1 + individual_hazard2) * delta_t)

                if e_test[i] == 1 and t == t_test[i]:
                    cif_event1.append(cif_event1_value)
                    cif_event2.append(0)
                elif e_test[i] == 2 and t == t_test[i]:
                    cif_event1.append(0)
                    cif_event2.append(cif_event2_value)
                else:
                    cif_event1.append(cif_event1_value)
                    cif_event2.append(cif_event2_value)
            else:
                cif_event1.append(cif_event1[-1] if j > 0 else 0)
                cif_event2.append(cif_event2[-1] if j > 0 else 0)

        cumulative_incidence_probs_event1.append(np.cumsum(cif_event1))
        cumulative_incidence_probs_event2.append(np.cumsum(cif_event2))

    return np.array(cumulative_incidence_probs_event1), np.array(cumulative_incidence_probs_event2)
# Calculate mean cumulative incidence probabilities
mean_cif_event1 = np.mean(cumulative_incidence_probs_event1, axis=0)
mean_cif_event2 = np.mean(cumulative_incidence_probs_event2, axis=0)

# Plotting the mean curves
plt.figure(figsize=(10, 6))
plt.plot(time_points, mean_cif_event1, label='Event 1', linestyle='-')
plt.plot(time_points, mean_cif_event2, label='Event 2', linestyle='--')

plt.xlabel('Time')
plt.ylabel('Cumulative Incidence Probability')
plt.title('Cumulative Incidence Curves for Test Data Samples')
plt.legend()
plt.show()